# Imports

In [308]:
import pandas as pd
import numpy as np
import igraph as ig
from collections import defaultdict



# Part 1

In [147]:
df = pd.read_csv("data/Part_A/1/balanced_graph.csv")
g = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
g.es["sign"] = df["sign"].tolist()
triangles = g.cliques(min=3, max=3)

In [148]:
zero_edges = [e.index for e in g.es if e["sign"] == 0]
triangle_edges = []
for tri in triangles:
    n1, n2, n3 = tri
    e1 = g.get_eid(n1, n2)
    e2 = g.get_eid(n2, n3)
    e3 = g.get_eid(n3, n1)
    triangle_edges.append((e1, e2, e3))



In [149]:
def is_triangle_balanced(tri_edges, graph):
    prod = 1
    for e in tri_edges:
        s = graph.es[e]["sign"]
        if s == 0:
            return True  # unknown edges can still be assigned
        prod *= s
    return prod > 0


In [150]:
def backtrack_balance(graph, zero_edges, triangle_edges, idx=0):
    if idx == len(zero_edges):
        for tri in triangle_edges:
            if not is_triangle_balanced(tri, graph):
                return False
        return True

    edge_idx = zero_edges[idx]

    for sign in [1, -1]:        
        graph.es[edge_idx]["sign"] = sign        
        valid = True
        for tri in triangle_edges:
            if edge_idx in tri and not is_triangle_balanced(tri, graph):
                valid = False
                break
        if valid:
            if backtrack_balance(graph, zero_edges, triangle_edges, idx + 1):
                return True        
        graph.es[edge_idx]["sign"] = 0
    return False


In [151]:
success = backtrack_balance(g, zero_edges, triangle_edges)
if success:
    print("Graph successfully balanced!")
else:
    print("No assignment can fully balance the graph with given constraints.")


Graph successfully balanced!


# Part B

In [309]:
def analyze_structural_balance(graph, graph_name):
    print(f"\n--- Analyzing: {graph_name} ---")
    
    g_pos = graph.copy()
    neg_edge_indices = [e.index for e in graph.es if e['sign'] == -1]
    g_pos.delete_edges(neg_edge_indices)
    clusters = g_pos.components()
    membership = clusters.membership
    num_supernodes = len(clusters)
    print(f"identified {num_supernodes} super-nodes (factions).")
    
    neg_edges = [e for e in graph.es if e['sign'] == -1]
    for edge in neg_edges:
        u, v = edge.tuple
        if membership[u] == membership[v]:
            u_name = graph.vs[u]['name']
            v_name = graph.vs[v]['name']
            print("Result: UNBALANCED")
            print(f"Reason: Internal contradiction. Node '{u_name}' and '{v_name}' are in the same super-node but have a negative edge.")
            print_supernode_assignments(graph, membership)
            return False

    
    reduced_graph = ig.Graph(num_supernodes)
    
    reduced_edges = []
    for edge in neg_edges:
        u, v = edge.tuple
        u_super = membership[u]
        v_super = membership[v]
        
        if u_super != v_super:
            reduced_edges.append((u_super, v_super))
            
    reduced_graph.add_edges(reduced_edges)
    reduced_graph.simplify()
        
    is_bipartite = reduced_graph.is_bipartite()
    
    if is_bipartite:
        print("Result: BALANCED")
        print("Reason: Reduced graph is bipartite (no odd cycles of negative edges).")
    else:
        print("Result: UNBALANCED")
        print("Reason: Reduced graph contains an odd cycle (contradiction in faction relations).")
        
    print_supernode_assignments(graph, membership)
    return is_bipartite

def print_supernode_assignments(graph, membership):
    print("\nSuper-node Assignments:")
    groups = defaultdict(list)
    for node_idx, cluster_id in enumerate(membership):        
        node_label = graph.vs[node_idx]['name'] 
        groups[cluster_id].append(node_label)
    
    for cluster_id, nodes in sorted(groups.items()):
        print(f"  Super-node {cluster_id}: {nodes}")

In [310]:
for i in ["a","b","c","d","e","f","g","h"]:
    df = pd.read_csv(f"data/Part_A/2/network_{i}.csv")
    graph = ig.Graph.TupleList(df[["u","v"]].itertuples(index=False), directed=False)
    graph.es["sign"] = df["sign"].tolist()
    analyze_structural_balance(graph,f"network_{i}")




--- Analyzing: network_a ---
identified 3 super-nodes (factions).
Result: BALANCED
Reason: Reduced graph is bipartite (no odd cycles of negative edges).

Super-node Assignments:
  Super-node 0: [0, 2, 4, 6, 9, 27, 29, 1, 22, 26, 15, 31, 32, 33, 34, 3, 7, 8, 19, 24, 12, 20, 5, 16, 25, 11, 30, 13]
  Super-node 1: [10, 14, 21, 23]
  Super-node 2: [17, 18, 28]

--- Analyzing: network_b ---
identified 3 super-nodes (factions).
Result: UNBALANCED
Reason: Internal contradiction. Node '0' and '14' are in the same super-node but have a negative edge.

Super-node Assignments:
  Super-node 0: [0, 3, 7, 9, 11, 14, 21, 28, 30, 34, 35, 36, 1, 13, 19, 32, 2, 5, 24, 20, 26, 27, 29, 31, 4, 37, 18, 10, 25, 33, 8, 12, 17, 23, 16, 22]
  Super-node 1: [6]
  Super-node 2: [15]

--- Analyzing: network_c ---
identified 3 super-nodes (factions).
Result: UNBALANCED
Reason: Reduced graph contains an odd cycle (contradiction in faction relations).

Super-node Assignments:
  Super-node 0: [0, 8, 9, 11, 28, 31, 6,